# 🧠 AI Mental Health Companion — Intent-Aware & Context-Grounded Architecture

Welcome to the updated **MindCare AI** companion notebook!  
This notebook breaks down the entire project pipeline into clear, modular **cells** featuring **Intent Detection & Conditional RAG Retrieval**.

### Modern Architecture Flow:
```
                    User Message
                         │
                    Safety Check
                         │
                  Intent Detection
                         │
              ┌──────────┴──────────┐
              │                     │
       Casual / Emotional       Information Request
              │                     │
              ↓                     ↓
         LLM + Memory          RAG Retriever
              │                     │
              └──────────┬──────────┘
                         │
                        LLM
                         │
                  Output Parser (Pydantic Schema)
                         │
                      Response
```

--- 
## 📌 Milestone 1: Environment Setup & Package Imports

In [ ]:
import os
import glob
import json
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

print("✅ Standard libraries loaded successfully!")

--- 
## 📌 Milestone 2 & 3: Knowledge Base Loading & Document Chunking

In [ ]:
from src.rag_pipeline import MentalHealthRAG

rag = MentalHealthRAG()
print(f"📚 Loaded {len(rag.documents)} Knowledge Base documents, indexed into {len(rag.chunks)} chunks.")

--- 
## 📌 Milestone 4: Intent Detection Engine

In [ ]:
from src.llm_chain import IntentDetector

detector = IntentDetector()

test_messages = [
    "I feel tired today.",
    "What is sleep hygiene and how can it help?",
    "I had a really hard day at university.",
    "How do I practice 4-7-8 breathing?"
]

for msg in test_messages:
    intent = detector.detect_intent(msg)
    print(f"Message: '{msg}' --> Detected Intent: [{intent.upper()}]")

--- 
## 📌 Milestone 5: Pydantic Output Parser (Adaptive Schema)

In [ ]:
from src.llm_chain import MentalHealthResponseSchema

# Casual venting schema example
casual_res = MentalHealthResponseSchema(
    response="I'm sorry you're feeling tired today. Have you been getting enough rest, or has it been an exhausting day?",
    intent="casual_emotional",
    topic=None,
    suggested_activity=None,
    sources=[],
    safety_level="normal"
)

print("💬 Casual Venting JSON Output:")
print(json.dumps(casual_res.model_dump(), indent=2))

# Factual info request schema example
info_res = MentalHealthResponseSchema(
    response="Sleep hygiene involves keeping a consistent schedule, optimizing your bedroom environment, and avoiding blue light before bed.",
    intent="information_request",
    topic="Sleep Hygiene & Relaxation",
    suggested_activity="Screen-free Wind Down Routine",
    sources=["Sleep Hygiene"],
    safety_level="normal"
)

print("\n📖 Information Request JSON Output:")
print(json.dumps(info_res.model_dump(), indent=2))

--- 
## 📌 Milestone 6: Safety Layer Evaluation

In [ ]:
from src.safety import SafetyChecker

safety = SafetyChecker()
is_risk, level, payload = safety.evaluate_input("I feel like ending it all")
print(f"Safety Check Result -> High Risk: {is_risk} | Level: {level}")
if is_risk:
    print("Crisis Response:", payload["response"])

--- 
## 📌 Milestone 7 - 16: End-to-End Intent-Aware Execution

In [ ]:
from src.llm_chain import MindCareAssistant

assistant = MindCareAssistant(rag_engine=rag)

queries = [
    "I feel tired today.",
    "What is sleep hygiene?",
    "I'm stressed about exams.",
    "How can I practice box breathing?"
]

for q in queries:
    print("=" * 60)
    print(f"👤 USER: {q}")
    res = assistant.generate_response(q)
    print(f"⚡ INTENT DETECTED: {res.intent.upper()}")
    print(f"💬 RESPONSE: {res.response}")
    if res.topic:
        print(f"🏷️ TOPIC: {res.topic}")
    if res.suggested_activity:
        print(f"🧘 ACTIVITY: {res.suggested_activity}")
    if res.sources:
        print(f"📚 SOURCES: {', '.join(res.sources)}")
    print("=" * 60 + "\n")